# LDA

Ajusta LDA con seis tópicos sobre el texto lematizado, en la corrida de referencia y sobre diez semillas.

## Rutas

Las rutas son relativas a la carpeta `notebooks/`. Los datos no forman parte del repositorio (ver README).

In [ ]:
# Comprobación de rutas
import os
PROY = os.path.abspath('..')
print("carpeta del notebook :", os.getcwd())
print("raiz del proyecto    :", PROY)
for _sub in ("Topic/Dataframes", "Topic/Resultados", "csv_parques"):
    _p = os.path.join(PROY, _sub)
    print(f"  {'OK ' if os.path.isdir(_p) else 'FALTA'}  {_sub}")
print("\nSi algo dice FALTA, el kernel no arranco en la carpeta del notebook,\n"
      "o los datos no estan donde se los espera (ver README).")

### Corrida de referencia (Tablas 4.3 a 4.5; Figuras 4.1, 4.2, 4.4 y 4.5)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import spacy
import warnings
import unicodedata
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from nltk.corpus import stopwords
from gensim.models.coherencemodel import CoherenceModel
import gensim.corpora as corpora
import nltk
from tqdm import tqdm
import pickle

warnings.filterwarnings("ignore", category=FutureWarning)
nltk.download('stopwords', quiet=True)


def remove_accents(text):
    if not isinstance(text, str):
        return text
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )


# 1. CARGA Y VALIDACIÓN DE DATOS

df = pd.read_pickle("../df_generic_cleaned.pkl")

# Determinar columna de texto fuente (la que ya tiene limpieza previa sin lematizar)
columna_texto_fuente = 'texto_limpio' if 'texto_limpio' in df.columns else 'text'
print(f"Datos cargados: {len(df)} reseñas.")
print(f"Columna fuente: {columna_texto_fuente}")


# 2. LEMATIZACIÓN CON SPACY

nlp = spacy.load("es_core_news_sm")
# Desactivar componentes innecesarios para acelerar (~3x más rápido)
nlp.disable_pipes(["ner", "parser"])

stop_words_base = set(stopwords.words('spanish'))


def lematizar_texto(texto):
    if not isinstance(texto, str) or not texto.strip():
        return ""
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúñü\s]', '', texto)
    doc = nlp(texto)
    tokens_limpios = []
    for token in doc:
        lema = remove_accents(token.lemma_.lower())
        # Chequear stopwords contra el lema, no contra token.text
        if lema not in stop_words_base and len(lema) > 2 and not token.is_punct:
            tokens_limpios.append(lema)
    return " ".join(tokens_limpios)


print("Lematizando reseñas...")
tqdm.pandas(desc="Lematización")
df['texto_lematizado'] = df[columna_texto_fuente].progress_apply(lematizar_texto)

# Eliminar reseñas que quedaron vacías después de lematizar
n_vacias = (df['texto_lematizado'].str.strip() == '').sum()
print(f"Reseñas vacías post-lematización: {n_vacias}")
df = df[df['texto_lematizado'].str.strip() != ''].copy()

# Verificación: antes vs después
print("Ejemplos de lematización:")
for i in range(min(3, len(df))):
    print(f"\n  ANTES:   {df.iloc[i][columna_texto_fuente][:120]}")
    print(f"  DESPUÉS: {df.iloc[i]['texto_lematizado'][:120]}")

columna_texto = 'texto_lematizado'

# ============================================================
# 3. STOPWORDS — FUENTE DE VERDAD (forma base / lematizada)
#    LDA y TF-IDF usan esto idéntico. W2V/BERTopic = mismos conceptos, flexionados.
#    Requisito: reseñas en inglés y portugués YA filtradas del corpus.
# ============================================================

CONTEXTO = {
    'lugar', 'sitio', 'espacio', 'zona', 'parte', 'parque', 'montevideo', 'uruguay',
    'visita', 'visitar', 'paseo', 'pasear', 'ir', 'hacer', 'dia', 'noche',
    'fin', 'semana', 'año', 'vez', 'momento', 'cosa', 'caso', 'tema', 'tipo',
    'aunque', 'sino', 'pues', 'tal', 'tan', 'asi', 'ahi', 'alla', 'aca',
    'pais', 'ciudad', 'capital', 'gente', 'persona', 'publico', 'mate', 'pasar',
    'quedar', 'estar', 'ser', 'haber', 'tener', 'poder', 'deber',
    'dar', 'ver', 'saber', 'querer', 'llegar', 'decir', 'salir', 'venir',
    'encontrar', 'dejar', 'poner', 'parecer', 'llevar', 'seguir',
    'etc', 'siempre', 'alguno', 'solo', 'medio', 'ahora', 'mismo',
}

SENTIMIENTO = {
    'lindo', 'hermoso', 'bonito', 'bello', 'precioso',
    'feo', 'horrible', 'espantoso', 'terrible',
    'bueno', 'malo', 'buen',
    'excelente', 'espectacular', 'genial', 'fantastico', 'increible',
    'maravilloso', 'divino', 'impecable', 'impresionante',
    'pesimo', 'lamentable', 'desastre', 'patetico',
    'mejor', 'peor', 'mayor', 'menor', 'gran', 'grande',
    'bien', 'mal', 'super', 'muy', 'bastante', 'poco', 'mucho',
    'agradable', 'desagradable', 'recomendable', 'recomendado',
    'tranquilo', 'paz', 'relax',
    'gustar', 'encantar', 'disfrutar', 'recomendar',
    'valer', 'ideal', 'perfecto', 'destacar', 'pena', 'disfrutable',
}

ERRORES = {
    'lindar',
    'exelente', 'exelentes', 'escelente',
    'imprecionante', 'ermoso', 'ermosa',
    'recomeir', 'recomiendable', 'recomendacion',
    'mas',
}

CORTESIA = {'gracias'}

LUGARES = {'prado'} # aparece muy mencionado en lso comentarios por ser un barrio, asi que lo saco.


PROTEGER = set()   
nombres_parque = set()
for nombre in df['nombre_parque'].str.lower().unique():
    for token in nombre.split():
        nombres_parque.add(remove_accents(token))
nombres_parque -= PROTEGER

# Lista final (forma base) — LDA y TF-IDF
final_stopwords = {remove_accents(w) for w in stopwords.words('spanish')}
for grupo in (CONTEXTO, SENTIMIENTO, ERRORES, CORTESIA, LUGARES):
    final_stopwords |= {remove_accents(w) for w in grupo}
final_stopwords |= nombres_parque
final_stopwords = sorted(final_stopwords)

# NOTA: 'aire' y 'libre' quedan FUERA a propósito (señal de "aire libre").

# --- Print de control ---
print(f"Stopwords totales: {len(final_stopwords)}")
print(f"\nNombres de parque incluidos ({len(nombres_parque)}):\n{sorted(nombres_parque)}")
print(f"\nLista completa:\n{final_stopwords}")


# 4. VECTORIZACIÓN (BAG OF WORDS)

print("\nVectorizando...")
vectorizer_lda = CountVectorizer(
    stop_words=final_stopwords,
    min_df=10,          # ignorar términos en menos de 10 documentos
    max_df=0.6,         # ignorar términos en más del 60% de los documentos
    ngram_range=(1, 2)  # unigramas y bigramas
)

matrix_lda = vectorizer_lda.fit_transform(df[columna_texto])
feature_names = vectorizer_lda.get_feature_names_out()
print(f"Matriz documento-término: {matrix_lda.shape}")
print(f"Vocabulario post-filtrado: {len(feature_names)} términos")


# 5. BÚSQUEDA DE K ÓPTIMO (Coherencia C_v + Perplejidad)

# Preparar corpus en formato gensim para coherencia C_v
textos_tokenizados = [doc.split() for doc in df[columna_texto]]
diccionario_gensim = corpora.Dictionary(textos_tokenizados)

rango_k = range(2, 10)
resultados_busqueda = []

print("BÚSQUEDA DE K ÓPTIMO")


for k in rango_k:
    lda_k = LatentDirichletAllocation(
        n_components=k,
        max_iter=50,
        learning_method='online',
        random_state=42,
        n_jobs=-1
    )
    lda_k.fit(matrix_lda)

    # Perplejidad
    perplexity = lda_k.perplexity(matrix_lda)

    # Coherencia C_v: extraer top palabras de cada tópico en formato gensim
    topicos_palabras = []
    for topic in lda_k.components_:
        top_indices = topic.argsort()[:-16:-1]
        topicos_palabras.append([feature_names[i] for i in top_indices])

    cm = CoherenceModel(
        topics=topicos_palabras,
        texts=textos_tokenizados,
        dictionary=diccionario_gensim,
        coherence='c_v'
    )
    coherencia = cm.get_coherence()

    resultados_busqueda.append({
        'k': k,
        'perplexity': perplexity,
        'coherence_cv': coherencia
    })
    print(f"  K={k:2d} | Perplejidad: {perplexity:>10,.1f} | Coherencia C_v: {coherencia:.4f}")

df_busqueda = pd.DataFrame(resultados_busqueda)

# Gráfica de selección de K
fig, ax1 = plt.subplots(figsize=(10, 5))

color_coh = 'tab:blue'
ax1.set_xlabel('Número de tópicos (K)', fontsize=12)
ax1.set_ylabel('Coherencia C_v', color=color_coh, fontsize=12)
ax1.plot(df_busqueda['k'], df_busqueda['coherence_cv'], 'o-', color=color_coh, linewidth=2, label='Coherencia C_v')
ax1.tick_params(axis='y', labelcolor=color_coh)

ax2 = ax1.twinx()
color_perp = 'tab:red'
ax2.set_ylabel('Perplejidad', color=color_perp, fontsize=12)
ax2.plot(df_busqueda['k'], df_busqueda['perplexity'], 's--', color=color_perp, linewidth=2, label='Perplejidad')
ax2.tick_params(axis='y', labelcolor=color_perp)

# Marcar K=6 con línea vertical para referencia
ax1.axvline(x=6, color='gray', linestyle=':', alpha=0.7, label='K=6 (fijado)')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.title('Selección de K: Coherencia C_v vs Perplejidad', fontsize=14)
fig.tight_layout()
fig.savefig("../Topic/Resultados/LDA_seleccion_k_optimo.png", dpi=300, bbox_inches="tight")
plt.show()

k_optimo_cv = int(df_busqueda.loc[df_busqueda['coherence_cv'].idxmax(), 'k'])
coherencia_en_6 = df_busqueda[df_busqueda['k'] == 6]['coherence_cv'].values[0]
coherencia_en_optimo = df_busqueda['coherence_cv'].max()

print(f"K óptimo por coherencia C_v: {k_optimo_cv} (C_v = {coherencia_en_optimo:.4f})")
print(f"Coherencia en K=6: {coherencia_en_6:.4f}")
print(f"Diferencia: {abs(coherencia_en_optimo - coherencia_en_6):.4f}")


# 6. MODELO FINAL CON K=6

n_temas = 6
print(f"Entrenando modelo final con K={n_temas}, max_iter=100...")

lda_model = LatentDirichletAllocation(
    n_components=n_temas,
    max_iter=100,           # suficiente para convergencia
    learning_method='online',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

lda_output = lda_model.fit_transform(matrix_lda)

log_likelihood = lda_model.score(matrix_lda)
perplexity_final = lda_model.perplexity(matrix_lda)
print(f"Log-likelihood final: {log_likelihood:,.1f}")
print(f"Perplejidad final: {perplexity_final:,.1f}")

# Coherencia del modelo final
topicos_final = []
for topic in lda_model.components_:
    top_indices = topic.argsort()[:-16:-1]
    topicos_final.append([feature_names[i] for i in top_indices])

cm_final = CoherenceModel(
    topics=topicos_final,
    texts=textos_tokenizados,
    dictionary=diccionario_gensim,
    coherence='c_v'
)
coherencia_final = cm_final.get_coherence()
print(f"Coherencia C_v del modelo final: {coherencia_final:.4f}")


# 7. VISUALIZACIÓN DE TÓPICOS

def plot_top_words_lda(model, feature_names, n_top_words=15, save_path=None):
    n_topics = model.n_components
    n_cols = 2
    n_rows = (n_topics + 1) // 2
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows), sharex=False)
    axes = axes.flatten()

    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7, color='teal')
        ax.set_title(f'Tópico {topic_idx}', fontdict={'fontsize': 14, 'fontweight': 'bold'})
        ax.invert_yaxis()
        ax.tick_params(axis='both', which='major', labelsize=10)

        print(f"  Tópico {topic_idx}: {', '.join(top_features[:10])}")

    # Ocultar ejes sobrantes si n_topics es impar
    for idx in range(n_topics, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle('Palabras clave por tópico (LDA)', fontsize=18, y=1.02)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


print("PALABRAS CLAVE POR TÓPICO:")
plot_top_words_lda(lda_model, feature_names, save_path="../Topic/Resultados/LDA_palabras_clave_topicos.png")


# 8. DISTRIBUCIÓN DE TEMAS POR PARQUE

df['tema_dominante'] = lda_output.argmax(axis=1)

# Guardar probabilidades por tópico (útil para análisis cruzados)
for i in range(n_temas):
    df[f'prob_topico_{i}'] = lda_output[:, i]

perfil_parques = pd.crosstab(
    df['nombre_parque'],
    df['tema_dominante'],
    normalize='index'
) * 100

print("\nDistribución de tópicos por parque (%):")
print(perfil_parques.round(1))

# Gráfica de barras apiladas
fig, ax = plt.subplots(figsize=(12, 8))
perfil_parques.plot(
    kind='barh',
    stacked=True,
    ax=ax,
    colormap='tab10',
    edgecolor='white'
)
plt.title('Distribución de tópicos por parque', fontsize=16)
plt.xlabel('% de reseñas', fontsize=12)
plt.ylabel('')
plt.legend(title='Tópico LDA', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
fig.savefig("../Topic/Resultados/LDAdistribucion_topicos_por_parque.png", dpi=300, bbox_inches="tight")
plt.show()


# 9. INSPECCIÓN CUALITATIVA (reseñas representativas por tópico)

# Para cada tópico se muestran las N reseñas con mayor probabilidad de pertenecer a ese tópico.

print("RESEÑAS MÁS REPRESENTATIVAS POR TÓPICO")

n_reseñas_ejemplo = 5
for t in range(n_temas):
    print(f"\n{'─' * 60}")
    print(f"TÓPICO {t} — Palabras clave: {', '.join(topicos_final[t][:8])}")
    print(f"{'─' * 60}")
    indices_top = lda_output[:, t].argsort()[::-1][:n_reseñas_ejemplo]
    for rank, idx in enumerate(indices_top, 1):
        # Mostrar el texto ORIGINAL (no lematizado) para legibilidad
        texto_original = df.iloc[idx][columna_texto_fuente][:250]
        prob = lda_output[idx, t]
        parque = df.iloc[idx]['nombre_parque']
        print(f"  [{rank}] p={prob:.3f} | {parque}")
        print(f"       \"{texto_original}...\"")


# 10. COHERENCIA POR TÓPICO INDIVIDUAL

print("COHERENCIA POR TÓPICO INDIVIDUAL")

for t_idx, t_words in enumerate(topicos_final):
    cm_individual = CoherenceModel(
        topics=[t_words],
        texts=textos_tokenizados,
        dictionary=diccionario_gensim,
        coherence='c_v'
    )
    coh = cm_individual.get_coherence()
    print(f"  Tópico {t_idx}: C_v = {coh:.4f} | {', '.join(t_words[:6])}")



df[['id', 'nombre_parque', 'text', 'tema_dominante'] +
   [f'prob_topico_{i}' for i in range(6)]
].to_pickle("../Topic/Dataframes/df_lda_final.pkl")

# 11. RESUMEN DE MÉTRICAS


print("RESUMEN")
print(f"Corpus: {len(df)} reseñas")
print(f"Preprocesamiento: lematización con spaCy (es_core_news_sm)")
print(f"Vocabulario post-filtrado: {len(feature_names)} términos")
print(f"Stopwords aplicadas: {len(final_stopwords)} (base + contexto + sentimiento)")
print(f"Vectorización: CountVectorizer, min_df=10, max_df=0.6, ngram_range=(1,2)")
print(f"Rango de K evaluado: {list(rango_k)}")
print(f"K óptimo por coherencia C_v: {k_optimo_cv} (C_v = {coherencia_en_optimo:.4f})")
print(f"K utilizado: {n_temas} (consistencia entre técnicas)")
print(f"Coherencia C_v en K={n_temas}: {coherencia_final:.4f}")
print(f"Perplejidad en K={n_temas}: {perplexity_final:,.1f}")
print(f"Log-likelihood: {log_likelihood:,.1f}")
print(f"Iteraciones LDA (modelo final): 100")
print(f"Método de aprendizaje: online")
print(f"Random state: 42")


# Guardar las metricas



analyzer = vectorizer_lda.build_analyzer()
fn = vectorizer_lda.get_feature_names_out()
topic_words_raw = [[fn[i] for i in t.argsort()[::-1][:100]] for t in lda_model.components_]
ref_texts = [analyzer(doc) for doc in df[columna_texto]]
pickle.dump({'topic_words_raw': topic_words_raw, 'ref_texts': ref_texts, 'final_stopwords': final_stopwords},
            open("../Topic/Dataframes/ckpt_lda.pkl", "wb"))
print("ckpt_lda.pkl OK (ref con analyzer)")

### Coherencia C_v de la corrida de referencia

In [ ]:
# === Coherencia C_v — LDA ===
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

TOP_N = 15
feature_names = vectorizer_lda.get_feature_names_out()
topic_words = []
for topic in lda_model.components_:
    palabras = [feature_names[i] for i in topic.argsort()[::-1]]
    palabras = [w for w in palabras if w not in final_stopwords][:TOP_N]
    topic_words.append(palabras)

textos_tokenizados = [doc.split() for doc in df[columna_texto]]
diccionario = Dictionary(textos_tokenizados)

cm = CoherenceModel(topics=topic_words, texts=textos_tokenizados,
                    dictionary=diccionario, coherence='c_v')
print(f"Coherencia C_v (LDA): {cm.get_coherence():.4f}")
for i, s in enumerate(cm.get_coherence_per_topic()):
    print(f"  Topico {i}: {s:.4f}")

    

### Diez semillas y barrido de K (Tablas 3.2, 4.1 y 4.2)

In [ ]:
# ============================================================
# MULTI-SEMILLA — LDA  (autocontenido)
# Puede ejecutarse sin la celda anterior.
# En una sola pasada por semilla hace el BARRIDO K=2..9 (criterio C_v, robustez de la
# seleccion de K) y captura el TOP-100 en K=6 (para la tabla de calidad C_npmi+TD).
# Cachea el corpus lematizado en prep_lemmatizado.pkl (compartido con TF-IDF) -> la
# lematizacion se paga UNA sola vez entre ambos notebooks.
# ============================================================
import pickle, os, re, unicodedata, numpy as np, pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from nltk.corpus import stopwords
import nltk, spacy
from tqdm import tqdm
nltk.download('stopwords', quiet=True)

BD    = ".."
DFDIR = f"{BD}/Topic/Dataframes"
SEEDS = [42, 0, 1, 2, 3, 4, 5, 6, 7, 8]   # 42 = baseline
K_RANGE = range(2, 10)                     # mismo rango que el barrido original
K_FINAL = 6                                # compromiso IMPUESTO por comparabilidad (NO optimo)

def remove_accents(text):
    if not isinstance(text, str): return text
    return ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')

# --- corpus lematizado: cache o construir (identico a la celda LDA/TF-IDF) ---
PREP = f"{DFDIR}/prep_lemmatizado.pkl"
if os.path.exists(PREP):
    d = pickle.load(open(PREP, 'rb')); corpus = d['corpus']; final_stopwords = d['final_stopwords']
    print(f"prep_lemmatizado.pkl cargado | {len(corpus)} docs | {len(final_stopwords)} stopwords")
else:
    df = pd.read_pickle(f"{BD}/df_generic_cleaned.pkl")
    col = 'texto_limpio' if 'texto_limpio' in df.columns else 'text'
    nlp = spacy.load("es_core_news_sm"); nlp.disable_pipes(["ner", "parser"])
    stop_base = set(stopwords.words('spanish'))
    def lem(t):
        if not isinstance(t, str) or not t.strip(): return ""
        t = re.sub(r'[^a-záéíóúñü\s]', '', t.lower())
        out = []
        for tok in nlp(t):
            l = remove_accents(tok.lemma_.lower())
            if l not in stop_base and len(l) > 2 and not tok.is_punct: out.append(l)
        return " ".join(out)
    tqdm.pandas(desc="Lematizando")
    lemm = df[col].progress_apply(lem)
    corpus = [x for x in lemm if x.strip()]
    CONTEXTO = {
        'lugar','sitio','espacio','zona','parte','parque','montevideo','uruguay',
        'visita','visitar','paseo','pasear','ir','hacer','dia','noche',
        'fin','semana','año','vez','momento','cosa','caso','tema','tipo',
        'aunque','sino','pues','tal','tan','asi','ahi','alla','aca',
        'pais','ciudad','capital','gente','persona','publico','mate','pasar',
        'quedar','estar','ser','haber','tener','poder','deber',
        'dar','ver','saber','querer','llegar','decir','salir','venir',
        'encontrar','dejar','poner','parecer','llevar','seguir',
        'etc','siempre','alguno','solo','medio','ahora','mismo'}
    SENTIMIENTO = {
        'lindo','hermoso','bonito','bello','precioso','feo','horrible','espantoso','terrible',
        'bueno','malo','buen','excelente','espectacular','genial','fantastico','increible',
        'maravilloso','divino','impecable','impresionante','pesimo','lamentable','desastre','patetico',
        'mejor','peor','mayor','menor','gran','grande','bien','mal','super','muy','bastante','poco','mucho',
        'agradable','desagradable','recomendable','recomendado','tranquilo','paz','relax',
        'gustar','encantar','disfrutar','recomendar','valer','ideal','perfecto','destacar','pena','disfrutable'}
    ERRORES = {'lindar','exelente','exelentes','escelente','imprecionante','ermoso','ermosa',
        'recomeir','recomiendable','recomendacion','mas'}
    CORTESIA = {'gracias'}; LUGARES = {'prado'}; PROTEGER = set()
    nombres = set()
    for nombre in df['nombre_parque'].str.lower().unique():
        for t in nombre.split(): nombres.add(remove_accents(t))
    nombres -= PROTEGER
    final_stopwords = {remove_accents(w) for w in stopwords.words('spanish')}
    for g in (CONTEXTO, SENTIMIENTO, ERRORES, CORTESIA, LUGARES):
        final_stopwords |= {remove_accents(w) for w in g}
    final_stopwords |= nombres
    final_stopwords = sorted(final_stopwords)
    pickle.dump({'corpus': corpus, 'final_stopwords': final_stopwords}, open(PREP, 'wb'))
    print(f"prep_lemmatizado.pkl creado | {len(corpus)} docs | {len(final_stopwords)} stopwords")

# --- vectorizacion (identica a la celda LDA) ---
vec = CountVectorizer(stop_words=final_stopwords, min_df=10, max_df=0.6, ngram_range=(1, 2))
X = vec.fit_transform(corpus)
fn = vec.get_feature_names_out()
analyzer = vec.build_analyzer()
ref_analyzer = [analyzer(doc) for doc in corpus]   # bigramas (para C_npmi luego) - invariante a la semilla
ref_split    = [doc.split() for doc in corpus]     # unigramas (para el C_v del barrido, como el original)
dic_cv = Dictionary(ref_split)

barrido = {}          # seed -> {k: C_v}
topw_by_seed = {}     # seed -> top-100 por topico en K=6
for s in SEEDS:
    curva = {}
    for k in K_RANGE:
        m = LatentDirichletAllocation(n_components=k, max_iter=50, learning_method='online',
                                      random_state=s, n_jobs=-1)
        m.fit(X)
        tw15 = [[fn[i] for i in t.argsort()[:-16:-1]] for t in m.components_]   # top-15 sin filtrar (como original)
        curva[k] = CoherenceModel(topics=tw15, texts=ref_split, dictionary=dic_cv,
                                  coherence='c_v').get_coherence()
        if k == K_FINAL:
            topw_by_seed[s] = [[fn[i] for i in t.argsort()[::-1][:100]] for t in m.components_]
    barrido[s] = curva
    kbest = max(curva, key=curva.get)
    print(f"LDA seed={s:>2} | C_v " + " ".join(f"K{k}={curva[k]:.3f}" for k in K_RANGE) + f" | argmax=K{kbest}")

pickle.dump({'topic_words_raw_by_seed': topw_by_seed, 'ref_texts': ref_analyzer,
             'final_stopwords': final_stopwords, 'seeds': SEEDS},
            open(f"{DFDIR}/ckpt_lda_multiseed.pkl", "wb"))
pickle.dump({'barrido': barrido, 'criterio': 'C_v', 'k_range': list(K_RANGE), 'seeds': SEEDS},
            open(f"{DFDIR}/barrido_lda_multiseed.pkl", "wb"))
print("OK -> ckpt_lda_multiseed.pkl + barrido_lda_multiseed.pkl")